# StockVision AI — Notebook 04: Model Experiments & Validation

**Objective:** Train, validate, and compare all models using time-series-aware walk-forward validation.

**Models compared:**
- Naive Baseline (random walk)
- Linear Regression
- Ridge Regression
- Random Forest Regressor
- Gradient Boosting
- XGBoost Regressor
- Logistic Regression (direction)
- XGBoost Classifier (direction)

**Validation strategy:** Expanding-window walk-forward (`TimeSeriesSplit`)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier

from src.utils.config import DATA_PROCESSED_DIR, TRAIN_END_DATE, VALIDATION_END_DATE
from src.models.evaluate import evaluate_regression, evaluate_classification, walk_forward_evaluation
from src.models.train import get_feature_columns, EXCLUDE_FROM_FEATURES

plt.style.use('dark_background')
pd.set_option('display.float_format', '{:.5f}'.format)

FOCUS_TICKER = 'TCS.NS'
TARGET_REG   = 'target_return_1d'
TARGET_CLF   = 'target_direction_1d'

## 1. Load Feature Data & Split

In [ ]:
path = DATA_PROCESSED_DIR / f"{FOCUS_TICKER.replace('.','_')}_features.parquet"
df = pd.read_parquet(path)
df['trade_date'] = pd.to_datetime(df['trade_date'])
df = df.sort_values('trade_date').reset_index(drop=True)

feature_cols = get_feature_columns(df)
df_clean = df.dropna(subset=feature_cols + [TARGET_REG, TARGET_CLF]).copy()

# Chronological split
train_df = df_clean[df_clean['trade_date'] <= TRAIN_END_DATE]
val_df   = df_clean[(df_clean['trade_date'] > TRAIN_END_DATE) & (df_clean['trade_date'] <= VALIDATION_END_DATE)]
test_df  = df_clean[df_clean['trade_date'] > VALIDATION_END_DATE]

X_train = train_df[feature_cols].values
y_train_reg = train_df[TARGET_REG].values
y_train_clf = train_df[TARGET_CLF].values.astype(int)

X_test  = test_df[feature_cols].values
y_test_reg = test_df[TARGET_REG].values
y_test_clf = test_df[TARGET_CLF].values.astype(int)

print(f'Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}')
print(f'Feature count: {len(feature_cols)}')
print(f'Train period: {train_df["trade_date"].min().date()} → {train_df["trade_date"].max().date()}')
print(f'Test period:  {test_df["trade_date"].min().date()} → {test_df["trade_date"].max().date()}')

## 2. Naive Baseline (Random Walk)

In [ ]:
# Naive baseline: predict zero return (most common in literature)
naive_pred_reg = np.zeros(len(y_test_reg))

# Or: predict previous day's return
naive_prev_day = test_df['daily_return_lag1'].values if 'daily_return_lag1' in test_df.columns else naive_pred_reg

baseline_metrics = evaluate_regression(y_test_reg, naive_pred_reg)
prev_day_metrics = evaluate_regression(y_test_reg, naive_prev_day)

# Majority class classifier baseline
majority_class = int(np.round(y_train_clf.mean()))
naive_clf_pred = np.full(len(y_test_clf), majority_class)
baseline_clf_metrics = evaluate_classification(y_test_clf, naive_clf_pred)

print('=== Naive Baselines ===')
print(f'Zero-return baseline   MAE: {baseline_metrics["mae"]:.5f}  Dir Acc: {baseline_metrics["directional_accuracy"]:.2%}')
print(f'Previous-return baseline MAE: {prev_day_metrics["mae"]:.5f}  Dir Acc: {prev_day_metrics["directional_accuracy"]:.2%}')
print(f'Majority-class clf accuracy: {baseline_clf_metrics["accuracy"]:.2%}')
print()
print('📌 Any model must beat these baselines to have practical value.')
print(f'   Positive class rate: {y_test_clf.mean():.2%} (markets go up ~{y_test_clf.mean()*100:.0f}% of days)')

## 3. Train All Regression Models

In [ ]:
import copy

reg_models = {
    'Naive Baseline':       None,  # handled separately
    'Linear Regression':    Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge Regression':     Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Random Forest':        RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_split=20, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    'XGBoost':              XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42, verbosity=0),
}

reg_results = []
trained_models = {}

# Naive first
naive_m = evaluate_regression(y_test_reg, naive_pred_reg)
naive_m['model'] = 'Naive Baseline'
reg_results.append(naive_m)

for name, model in reg_models.items():
    if model is None:
        continue
    print(f'Training {name}...', end=' ', flush=True)
    m = copy.deepcopy(model)
    m.fit(X_train, y_train_reg)
    y_pred = m.predict(X_test)
    metrics = evaluate_regression(y_test_reg, y_pred, naive_pred=naive_pred_reg)
    metrics['model'] = name
    reg_results.append(metrics)
    trained_models[name] = (m, y_pred)
    print(f'MAE={metrics["mae"]:.5f}  Dir_Acc={metrics["directional_accuracy"]:.2%}')

reg_df = pd.DataFrame(reg_results)[['model','mae','rmse','r_squared','directional_accuracy','baseline_mae','improvement_over_baseline']]
print('\n=== Regression Leaderboard ===')
display(reg_df.sort_values('mae'))

## 4. Walk-Forward Validation (XGBoost)

In [ ]:
print('Running walk-forward validation (4 folds)...')
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0)

wf_results = walk_forward_evaluation(
    df=df_clean,
    model=xgb_model,
    feature_cols=feature_cols,
    target_col=TARGET_REG,
    n_splits=4,
    task='regression'
)

print('\n=== Walk-Forward Results per Fold ===')
display(wf_results[['fold', 'train_size', 'test_size', 'mae', 'rmse', 'directional_accuracy']])

print(f'\nMean MAE across folds:              {wf_results["mae"].mean():.5f}')
print(f'Std  MAE across folds:              {wf_results["mae"].std():.5f}')
print(f'Mean Directional Accuracy:          {wf_results["directional_accuracy"].mean():.2%}')

# Plot MAE per fold
fig = px.bar(
    wf_results, x='fold', y='mae',
    title='XGBoost Walk-Forward MAE per Fold',
    template='plotly_dark', height=350,
    color='mae', color_continuous_scale='reds'
)
fig.add_hline(y=baseline_metrics['mae'], line_dash='dash', line_color='white',
              annotation_text='Naive Baseline')
fig.show()

## 5. Actual vs Predicted Returns

In [ ]:
best_model_name = 'XGBoost'
if best_model_name in trained_models:
    _, y_pred_best = trained_models[best_model_name]

    dates = pd.to_datetime(test_df['trade_date'].values)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=dates, y=y_test_reg * 100,
        name='Actual Return', line=dict(color='#69f0ae', width=1.5)
    ))
    fig.add_trace(go.Scatter(
        x=dates, y=y_pred_best * 100,
        name='Predicted Return', line=dict(color='#4fc3f7', width=1.5, dash='dash')
    ))
    fig.add_hline(y=0, line_dash='dot', line_color='gray', opacity=0.5)
    fig.update_layout(
        template='plotly_dark', height=400,
        title=f'{FOCUS_TICKER} — Actual vs Predicted Return (Test Period)',
        xaxis_title='Date', yaxis_title='Return (%)',
        hovermode='x unified'
    )
    fig.show()

    # Scatter plot: predicted vs actual
    fig2 = px.scatter(
        x=y_test_reg * 100, y=y_pred_best * 100,
        opacity=0.4, trendline='ols',
        title='Predicted vs Actual Return Scatter',
        labels={'x': 'Actual Return (%)', 'y': 'Predicted Return (%)'},
        template='plotly_dark', height=400
    )
    fig2.show()

## 6. Classification Models — Direction Prediction

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

clf_models = {
    'Majority Class Baseline': None,
    'Logistic Regression':  Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(C=0.1, max_iter=500, random_state=42, class_weight='balanced'))]),
    'Random Forest Clf':    RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight='balanced', n_jobs=-1),
    'XGBoost Clf':          XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0, eval_metric='logloss'),
}

clf_results = []

# Majority baseline
m_baseline = evaluate_classification(y_test_clf, naive_clf_pred)
m_baseline['model'] = 'Majority Class Baseline'
clf_results.append(m_baseline)

for name, model in clf_models.items():
    if model is None:
        continue
    print(f'Training {name}...', end=' ', flush=True)
    m = copy.deepcopy(model)
    m.fit(X_train, y_train_clf)
    y_pred = m.predict(X_test)
    y_prob = m.predict_proba(X_test)[:,1] if hasattr(m, 'predict_proba') else None
    metrics = evaluate_classification(y_test_clf, y_pred, y_prob)
    metrics['model'] = name
    clf_results.append(metrics)
    print(f'F1={metrics["f1_score"]:.4f}  AUC={metrics.get("roc_auc",0):.4f}  Acc={metrics["accuracy"]:.2%}')

clf_df = pd.DataFrame(clf_results)[['model','accuracy','precision_score','recall_score','f1_score','roc_auc']]
print('\n=== Classification Leaderboard ===')
display(clf_df.sort_values('f1_score', ascending=False))

## 7. Confusion Matrix (Best Classifier)

In [ ]:
# Refit XGBoost Clf for confusion matrix
xgb_clf = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0, eval_metric='logloss')
xgb_clf.fit(X_train, y_train_clf)
y_pred_clf = xgb_clf.predict(X_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_clf, y_pred_clf)

fig = px.imshow(
    cm,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=['DOWN (0)', 'UP (1)'],
    y=['DOWN (0)', 'UP (1)'],
    color_continuous_scale='Blues',
    text_auto=True,
    title='Confusion Matrix — XGBoost Classifier',
    template='plotly_dark', height=400
)
fig.show()

# Classification report
from sklearn.metrics import classification_report
print('=== Classification Report ===')
print(classification_report(y_test_clf, y_pred_clf, target_names=['DOWN', 'UP']))

## 8. Feature Importance — XGBoost

In [ ]:
# Refit XGBoost Regressor for feature importance
xgb_reg = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0)
xgb_reg.fit(X_train, y_train_reg)

fi_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_reg.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig = px.bar(
    fi_df, x='importance', y='feature', orientation='h',
    title='🏆 XGBoost Regressor — Top 20 Feature Importances',
    template='plotly_dark', height=550,
    color='importance', color_continuous_scale='viridis'
)
fig.update_layout(coloraxis_showscale=False, margin=dict(l=200))
fig.show()

print('Top 10 features driving XGBoost predictions:')
for _, row in fi_df.head(10).iterrows():
    print(f'  {row["feature"]:<40} {row["importance"]:.4f}')

## 9. Final Model Comparison Summary

In [ ]:
print('=' * 70)
print('STOCKVISION AI — MODEL EVALUATION SUMMARY')
print('=' * 70)
print(f'Ticker:          {FOCUS_TICKER}')
print(f'Target:          Next-day return (regression) & direction (classification)')
print(f'Training period: {train_df["trade_date"].min().date()} → {train_df["trade_date"].max().date()}')
print(f'Test period:     {test_df["trade_date"].min().date()} → {test_df["trade_date"].max().date()}')
print()

reg_sorted = reg_df.sort_values('mae')
best_reg = reg_sorted.iloc[0]
print('REGRESSION RESULTS')
print(f'  Best model:               {best_reg["model"]}')
print(f'  Test MAE:                 {best_reg["mae"]:.5f}')
print(f'  Naive baseline MAE:       {baseline_metrics["mae"]:.5f}')
if pd.notna(best_reg.get('improvement_over_baseline')):
    print(f'  Improvement over baseline: {best_reg["improvement_over_baseline"]:.1f}%')
print(f'  Directional accuracy:     {best_reg["directional_accuracy"]:.2%}')
print()

clf_sorted = pd.DataFrame(clf_results).sort_values('f1_score', ascending=False)
best_clf = clf_sorted.iloc[0]
print('CLASSIFICATION RESULTS')
print(f'  Best model:               {best_clf["model"]}')
print(f'  F1-Score:                 {best_clf["f1_score"]:.4f}')
print(f'  ROC-AUC:                  {best_clf.get("roc_auc", "N/A")}')
print(f'  Accuracy:                 {best_clf["accuracy"]:.2%}')
print()
print('IMPORTANT LIMITATION:')
print('  Historical market patterns may not remain stable under')
print('  changing market conditions. These forecasts are for')
print('  educational and analytical purposes only.')
print('=' * 70)